# Explore the raw feed, column by column

**Goal:** build a mental model of the two parquet files before deciding anything.

**Plan:**
1. Land in the data. Check dimensions and first rows.
2. Walk the outer delivery-metadata columns one group at a time. Name each column. Show how it varies.
3. Open the `payload_json` string. It carries the real Open Finance API response. Explore one shape per `(investment_type, payload_kind)`.
4. Do the same for `transaction_json`.
5. End with a scratch-pad of observations for the design docs.

**Spec reference** (keep open in another tab):
- Portal: https://openfinancebrasil.atlassian.net/wiki/spaces/OF/overview
- Investments Swagger UI: https://openbanking-brasil.github.io/openapi/swagger-apis/investments/?urls.primaryName=1.0.1
- Raw YAMLs on GitHub: https://github.com/OpenBanking-Brasil/openapi/tree/main/swagger-apis


## 1 · Landing


In [1]:
import duckdb, json, pandas as pd
from pathlib import Path

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 40)

RAW_DATA_DIR = Path('../data/raw')
duckdb_conn = duckdb.connect()
duckdb_conn.execute(f"CREATE VIEW raw_positions    AS SELECT * FROM read_parquet('{RAW_DATA_DIR/'raw_positions.parquet'}')")
duckdb_conn.execute(f"CREATE VIEW raw_transactions AS SELECT * FROM read_parquet('{RAW_DATA_DIR/'raw_transactions.parquet'}')")


def sql(sql, params=None):
    return duckdb_conn.execute(sql, params or []).df()

def sql_scalar(sql, params=None):
    return duckdb_conn.execute(sql, params or []).fetchone()[0]

position_count    = sql_scalar('SELECT count(*) FROM raw_positions')
transaction_count = sql_scalar('SELECT count(*) FROM raw_transactions')
print(f'positions:    {position_count:>8,} rows')
print(f'transactions: {transaction_count:>8,} rows')

positions:     210,647 rows
transactions:  321,117 rows


### Both schemas are all VARCHAR

Every column is `VARCHAR`, including timestamps and identifiers. Typing waits for the staging layer.

The two files differ:
- Positions carries `payload_kind` and `payload_json`.
- Transactions carries `payload_source`, `page`, `query_window`, and `transaction_json`.


In [2]:
print('--- POSITIONS ---');    print(sql('DESCRIBE raw_positions').to_string(index=False))
print('\n--- TRANSACTIONS ---'); print(sql('DESCRIBE raw_transactions').to_string(index=False))

--- POSITIONS ---
        column_name column_type null  key default extra
     institution_id     VARCHAR  YES None    None  None
   institution_name     VARCHAR  YES None    None  None
           party_id     VARCHAR  YES None    None  None
         account_id     VARCHAR  YES None    None  None
      connection_id     VARCHAR  YES None    None  None
        snapshot_id     VARCHAR  YES None    None  None
snapshot_created_at     VARCHAR  YES None    None  None
      investment_id     VARCHAR  YES None    None  None
    investment_type     VARCHAR  YES None    None  None
       payload_kind     VARCHAR  YES None    None  None
             s3_uri     VARCHAR  YES None    None  None
       payload_json     VARCHAR  YES None    None  None
        ingested_at     VARCHAR  YES None    None  None

--- TRANSACTIONS ---
        column_name column_type null  key default extra
     institution_id     VARCHAR  YES None    None  None
   institution_name     VARCHAR  YES None    None  None
        

### Eyeball a few raw rows

The JSON columns show as long strings. That is fine. We open them below.


In [3]:
sql('SELECT * FROM raw_positions LIMIT 3')

,institution_id,institution_name,party_id,account_id,connection_id,snapshot_id,snapshot_created_at,investment_id,investment_type,payload_kind,s3_uri,payload_json,ingested_at
0,00000000000001,Nubank,a3990e1e-7ce8-54c9-a079-3b46f0e49411,f878d8d8-7166-5816-9e84-609629bdd36e,33df6586-1f47-582e-94b5-f5f60680e43c,9e246f99-6d8d-5e76-aab3-ee80ec0dcf50,2026-07-30T09:00:01Z,f03d8ad0-160f-5cb5-b383-6e455c760a05,TREASURE_TITLES,detail,s3://of-snapshots-sample/prod/9e246f99-6d8d-5e76-aab3-ee80ec0dcf50/investments/treasure-titles/f03d8ad0-160f-5cb5-b3...,"{""data"":{""isinCode"":""BRSTNCLTN8C3"",""productName"":""Tesouro Prefixado 2027"",""remuneration"":{""indexer"":""PRE_FIXADO"",""ra...",2026-07-30T09:01:01Z
1,00000000000001,Nubank,a3990e1e-7ce8-54c9-a079-3b46f0e49411,f878d8d8-7166-5816-9e84-609629bdd36e,33df6586-1f47-582e-94b5-f5f60680e43c,9e246f99-6d8d-5e76-aab3-ee80ec0dcf50,2026-07-30T09:00:01Z,f03d8ad0-160f-5cb5-b383-6e455c760a05,TREASURE_TITLES,balances,s3://of-snapshots-sample/prod/9e246f99-6d8d-5e76-aab3-ee80ec0dcf50/investments/treasure-titles/f03d8ad0-160f-5cb5-b3...,"{""data"":{""referenceDateTime"":""2026-07-30T09:00:00Z"",""updatedUnitPrice"":{""amount"":""504.63"",""currency"":""BRL""},""grossAm...",2026-07-30T09:01:01Z
2,00000000000001,Nubank,a3990e1e-7ce8-54c9-a079-3b46f0e49411,f878d8d8-7166-5816-9e84-609629bdd36e,33df6586-1f47-582e-94b5-f5f60680e43c,9e246f99-6d8d-5e76-aab3-ee80ec0dcf50,2026-07-30T09:00:01Z,be6dd9c2-e4ca-518b-9966-5e2fc451ad72,CREDIT_FIXED_INCOMES,detail,s3://of-snapshots-sample/prod/9e246f99-6d8d-5e76-aab3-ee80ec0dcf50/investments/credit-fixed-incomes/be6dd9c2-e4ca-51...,"{""data"":{""issuerInstitutionCnpjNumber"":""92894922000108.00"",""isinCode"":""BRDEBSEC01A1"",""investmentType"":""DEBENTURES"",""...",2026-07-30T09:01:01Z


In [4]:
sql('SELECT * FROM raw_transactions LIMIT 3')

,institution_id,institution_name,party_id,account_id,connection_id,snapshot_id,snapshot_created_at,investment_id,investment_type,payload_source,page,s3_uri,transaction_json,query_window,ingested_at
0,00000000000001,Nubank,a3990e1e-7ce8-54c9-a079-3b46f0e49411,f878d8d8-7166-5816-9e84-609629bdd36e,33df6586-1f47-582e-94b5-f5f60680e43c,9e246f99-6d8d-5e76-aab3-ee80ec0dcf50,2026-07-30T09:00:01Z,f03d8ad0-160f-5cb5-b383-6e455c760a05,TREASURE_TITLES,transactions,1,s3://of-snapshots-sample/prod/9e246f99-6d8d-5e76-aab3-ee80ec0dcf50/investments/treasure-titles/f03d8ad0-160f-5cb5-b3...,"{""type"":""ENTRADA"",""transactionType"":""COMPRA"",""transactionDate"":""2026-03-12"",""transactionUnitPrice"":{""amount"":""500.00...",2025-07-30/2026-07-30,2026-07-30T09:01:01Z
1,00000000000001,Nubank,a3990e1e-7ce8-54c9-a079-3b46f0e49411,f878d8d8-7166-5816-9e84-609629bdd36e,33df6586-1f47-582e-94b5-f5f60680e43c,42fef1b2-7a82-5017-92aa-67662d678121,2026-08-02T09:00:02Z,f03d8ad0-160f-5cb5-b383-6e455c760a05,TREASURE_TITLES,transactions,1,s3://of-snapshots-sample/prod/42fef1b2-7a82-5017-92aa-67662d678121/investments/treasure-titles/f03d8ad0-160f-5cb5-b3...,"{""type"":""ENTRADA"",""transactionType"":""COMPRA"",""transactionDate"":""2026-03-12"",""transactionUnitPrice"":{""amount"":""500.00...",2025-08-02/2026-08-02,2026-08-02T09:06:02Z
2,00000000000001,Nubank,a3990e1e-7ce8-54c9-a079-3b46f0e49411,f878d8d8-7166-5816-9e84-609629bdd36e,33df6586-1f47-582e-94b5-f5f60680e43c,35b95095-d1b1-5252-9476-10b4c09b2f44,2026-08-05T09:00:03Z,f03d8ad0-160f-5cb5-b383-6e455c760a05,TREASURE_TITLES,transactions,1,s3://of-snapshots-sample/prod/35b95095-d1b1-5252-9476-10b4c09b2f44/investments/treasure-titles/f03d8ad0-160f-5cb5-b3...,"{""type"":""ENTRADA"",""transactionType"":""COMPRA"",""transactionDate"":""2026-03-12"",""transactionUnitPrice"":{""amount"":""500.00...",2025-08-05/2026-08-05,2026-08-05T09:07:03Z


## 2 · Outer columns are delivery metadata

The 15 outer columns are metadata that Decade's ingestion layer wraps around the raw provider payloads. They tell us *how* records arrive, before we look at what is inside.

We walk four groups:
- **Who:** the institution and the customer.
- **Where:** account and connection.
- **When:** snapshot and ingestion timestamps.
- **What:** investment identifiers and which endpoint served the payload.


### 2.1 · Who: `institution_id`, `institution_name`, `party_id`

- **`institution_id`:** Decade's identifier for the source bank. Should map 1:1 with `institution_name`.
- **`institution_name`:** human-readable. Not authoritative. Can drift.
- **`party_id`:** the customer. "Party" is Open Finance vocabulary (`PF` is a natural person, `PJ` is a legal entity).

Watch for multiple `institution_name` values per `institution_id` (rebrand or typo), or the reverse (bad joins upstream).


In [5]:
sql("""
  SELECT institution_id, institution_name,
         count(*)                 AS record_count,
         count(DISTINCT party_id) AS customer_count
  FROM raw_positions
  GROUP BY institution_id, institution_name
  ORDER BY record_count DESC
""")

,institution_id,institution_name,record_count,customer_count
0,00000000000001,Nubank,80567,160
1,00000000000003,Itau,39314,65
2,00000000000002,Banco XP S.A.,28558,79
3,00000000000004,BTG Banking,25468,62
4,00000000000006,C6 Bank,15828,36
5,00000000000005,Banco Inter PF,14536,42
6,00000000000007,PicPay,4118,22
7,00000000000008,Banco do Brasil,2258,17


In [6]:
# Are institution_id and institution_name in bijection? (they should be)
sql("""
  SELECT institution_id, count(DISTINCT institution_name) AS distinct_name_count
  FROM raw_positions
  GROUP BY institution_id
  HAVING count(DISTINCT institution_name) > 1
""")

,institution_id,distinct_name_count


In [7]:
# Customer scale : 'a few hundred synthetic customers'
sql("SELECT count(DISTINCT party_id) AS customer_count FROM raw_positions")

,customer_count
0,400


### 2.2 · Where: `account_id`, `connection_id`

- **`account_id`:** the investment account within the institution. One customer can hold accounts at multiple institutions, and multiple accounts at one institution.
- **`connection_id`:** Decade's identifier for the customer's *authorized connection* to that institution (an Open Finance consent). One consent can produce many snapshots over time.

Cardinality expectation: `customers ≤ connections ≤ accounts`.


In [8]:
sql("""
  SELECT
    count(DISTINCT party_id)                   AS customer_count,
    count(DISTINCT connection_id)              AS connection_count,
    count(DISTINCT account_id)                 AS account_count,
    count(DISTINCT (party_id, institution_id)) AS customer_institution_pair_count
  FROM raw_positions
""")

,customer_count,connection_count,account_count,customer_institution_pair_count
0,400,500,500,483


In [9]:
# How many accounts does an average customer have?
sql("""
  WITH accounts_by_customer AS (
    SELECT party_id, count(DISTINCT account_id) AS accounts_per_customer
    FROM raw_positions
    GROUP BY party_id
  )
  SELECT accounts_per_customer, count(*) AS customer_count
  FROM accounts_by_customer
  GROUP BY accounts_per_customer
  ORDER BY accounts_per_customer
""")

,accounts_per_customer,customer_count
0,1,316
1,2,71
2,3,10
3,4,3


### 2.3 · When: `snapshot_created_at`, `ingested_at`, `query_window` (transactions only)

- **`snapshot_created_at`:** when the provider built the *snapshot*. The natural per-snapshot event time.
- **`snapshot_id`:** groups all records that arrived together in one sync from one institution.
- **`ingested_at`:** when Decade's pipeline received and wrote the record. **This is the natural watermark for incremental jobs** (`arrival_time` in case-study parlance).
- **`query_window`** (transactions only): the date range the provider was asked for. Open Finance transaction endpoints take `fromTransactionDate` and `toTransactionDate`. This column records what we asked.


In [10]:
sql("""
  SELECT
    min(snapshot_created_at) AS earliest_snapshot,
    max(snapshot_created_at) AS latest_snapshot,
    min(ingested_at)         AS earliest_ingest,
    max(ingested_at)         AS latest_ingest
  FROM raw_positions
""")

,earliest_snapshot,latest_snapshot,earliest_ingest,latest_ingest
0,2026-07-28T09:00:32Z,2026-08-21T10:30:00Z,2026-07-28T09:05:45Z,2026-08-21T10:36:59Z


In [11]:
# Snapshot cadence per institution : the case study says 'irregular and differs by institution'
sql("""
  SELECT institution_name,
         count(DISTINCT snapshot_id) AS snapshot_count,
         min(snapshot_created_at)    AS first_snapshot,
         max(snapshot_created_at)    AS last_snapshot
  FROM raw_positions
  GROUP BY institution_name
  ORDER BY snapshot_count DESC
""")

,institution_name,snapshot_count,first_snapshot,last_snapshot
0,Nubank,2152,2026-07-28T09:06:32Z,2026-08-21T10:29:43Z
1,Itau,853,2026-07-28T09:05:38Z,2026-08-21T10:25:01Z
2,Banco XP S.A.,822,2026-07-28T09:04:45Z,2026-08-21T10:25:54Z
3,Banco Inter PF,719,2026-07-28T09:12:42Z,2026-08-21T10:29:19Z
4,BTG Banking,579,2026-07-28T09:00:32Z,2026-08-21T10:19:22Z
5,C6 Bank,254,2026-07-28T09:03:27Z,2026-08-21T10:30:00Z
6,PicPay,114,2026-07-28T09:01:08Z,2026-08-21T10:16:48Z
7,Banco do Brasil,93,2026-07-28T09:03:39Z,2026-08-21T10:26:01Z


In [12]:
# Gap between provider snapshot time and Decade ingest time : how 'late' are records?
sql("""
  WITH ingest_lag_seconds AS (
    SELECT institution_name,
           epoch(cast(ingested_at as timestamp)) - epoch(cast(snapshot_created_at as timestamp)) AS lag_seconds
    FROM raw_positions
  )
  SELECT institution_name,
         percentile_cont(0.5)  WITHIN GROUP (ORDER BY lag_seconds) / 60 AS median_lag_minutes,
         percentile_cont(0.95) WITHIN GROUP (ORDER BY lag_seconds) / 60 AS p95_lag_minutes
  FROM ingest_lag_seconds
  GROUP BY institution_name
  ORDER BY median_lag_minutes DESC
""")

,institution_name,median_lag_minutes,p95_lag_minutes
0,Banco do Brasil,6.0,9.0
1,Itau,5.0,9.0
2,Banco XP S.A.,5.0,9.0
3,Nubank,5.0,9.0
4,Banco Inter PF,5.0,9.0
5,C6 Bank,5.0,9.0
6,BTG Banking,5.0,9.0
7,PicPay,5.0,9.0


### 2.4 · What: `investment_id`, `investment_type`, `payload_kind`

- **`investment_id`:** the provider's identifier for one holding. **The case study warns that this identifier can churn.** The Open Finance spec (`variable-incomes/1.3.0.yml`) *mandates* reuse after 12-month dormancy, but institutions may not comply.
- **`investment_type`:** which product family. Maps 1:1 to the 5 sub-APIs of Open Finance Investments.
- **`payload_kind`** (positions only): which endpoint served the payload.
  - `balances` calls `/investments/{investmentId}/balances`. It returns snapshot values (quantity, market value).
  - `detail` calls `/investments/{investmentId}`. It returns the security's static identity (ISIN, ticker, CNPJ, `dueDate`).

**Modeling implication:** for each holding at each snapshot we receive **two records** (balance and detail). To get one logical position row, join them on `(institution_id, party_id, account_id, snapshot_id, investment_id)`. Detail is slowly-changing (`dueDate`, ISIN do not move). Balance changes every snapshot.


In [13]:
sql("""
  SELECT investment_type, payload_kind,
         count(*)                      AS record_count,
         count(DISTINCT investment_id) AS holding_count
  FROM raw_positions
  GROUP BY investment_type, payload_kind
  ORDER BY investment_type, payload_kind
""")

,investment_type,payload_kind,record_count,holding_count
0,BANK_FIXED_INCOMES,balances,66183,6303
1,BANK_FIXED_INCOMES,detail,64535,6143
2,CREDIT_FIXED_INCOMES,balances,5447,504
3,CREDIT_FIXED_INCOMES,detail,5447,504
4,FUNDS,balances,8355,762
5,FUNDS,detail,8355,762
6,TREASURE_TITLES,balances,9138,822
7,TREASURE_TITLES,detail,9138,822
8,VARIABLE_INCOMES,balances,17570,1611
9,VARIABLE_INCOMES,detail,16479,1509


In [14]:
# Verify: within a single snapshot, do we always get both balances and detail for each holding?
sql("""
  WITH records_per_kind AS (
    SELECT snapshot_id, investment_id, payload_kind, count(*) AS record_count
    FROM raw_positions
    GROUP BY snapshot_id, investment_id, payload_kind
  ),
  balances_side AS (
    SELECT snapshot_id, investment_id, record_count AS balances_per_holding
    FROM records_per_kind WHERE payload_kind = 'balances'
  ),
  detail_side AS (
    SELECT snapshot_id, investment_id, record_count AS details_per_holding
    FROM records_per_kind WHERE payload_kind = 'detail'
  )
  SELECT balances_per_holding, details_per_holding, count(*) AS holding_snapshot_count
  FROM balances_side FULL JOIN detail_side USING (snapshot_id, investment_id)
  GROUP BY balances_per_holding, details_per_holding
  ORDER BY holding_snapshot_count DESC
""")

,balances_per_holding,details_per_holding,holding_snapshot_count
0,1,1.0,103954
1,1,NaN,2739


### 2.5 · The odd columns: `snapshot_id`, `s3_uri`, `page`, `payload_source`

- **`snapshot_id`:** groups one sync from one institution for one customer. A snapshot contains N holdings × 2 payloads.
- **`s3_uri`:** provenance. Points to the raw JSON in blob storage. Useful for audit, not for querying.
- **`page`** (transactions only): the Open Finance transaction endpoints paginate. This column records which page the row came from.
- **`payload_source`** (transactions only): which endpoint served the record. `transactions` returns history. `transactions-current` returns the last 7 days only. Completeness and freshness guarantees differ.


In [15]:
sql("""
  SELECT payload_source, count(*) AS record_count
  FROM raw_transactions GROUP BY payload_source
""")

,payload_source,record_count
0,transactions,314685
1,transactions-current,6432


In [16]:
# Snapshots : one per (institution, customer, sync run)
sql("""
  SELECT
    count(DISTINCT snapshot_id)                        AS snapshot_count,
    count(*) / count(DISTINCT snapshot_id)::float      AS average_records_per_snapshot
  FROM raw_positions
""")

,snapshot_count,average_records_per_snapshot
0,5586,37.709808


## 3 · Open the JSON: position payloads

The payload structure differs per `(investment_type, payload_kind)`. Five families × two kinds gives 10 payload shapes. Each cell below picks a real sample, pretty-prints it, and adds per-field notes from the OFB spec.

Helper to pull a sample:


In [17]:
def sample_position_payload(investment_type: str, payload_kind: str) -> dict:
    row = duckdb_conn.execute(
        'SELECT payload_json FROM raw_positions WHERE investment_type = ? AND payload_kind = ? LIMIT 1',
        [investment_type, payload_kind]
    ).fetchone()
    return json.loads(row[0])['data']  # unwrap the {data, meta, links} envelope

def sample_transaction_payload(investment_type: str) -> dict:
    row = duckdb_conn.execute(
        'SELECT transaction_json FROM raw_transactions WHERE investment_type = ? LIMIT 1',
        [investment_type]
    ).fetchone()
    return json.loads(row[0])

def print_payload(label: str, payload: dict) -> None:
    print(f'--- {label} ---')
    print(json.dumps(payload, indent=2, ensure_ascii=False))

### 3.1 · `VARIABLE_INCOMES`: equities, ETF, FII

**Spec:** [variable-incomes/1.3.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/variable-incomes/1.3.0.yml)

**detail:** the security's static identity. Very short: ISIN, ticker, issuer CNPJ. This is why variable income is the easiest family to key logically. `(isinCode, ticker)` is highly deterministic because B3 standardizes both.

**balances:** daily snapshot. `closingPrice` is the price on `referenceDate` (D-1 or D-2 per the spec).


In [18]:
print_payload('detail', sample_position_payload('VARIABLE_INCOMES', 'detail'))

--- detail ---
{
  "issuerInstitutionCnpjNumber": "60872504000123",
  "isinCode": "BRITUBACNPR1",
  "ticker": "ITUB4"
}


### 3.2 · `FUNDS`: investment funds

**Spec:** [funds/1.1.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/funds/1.1.0.yml)

**detail:** the fund's CNPJ is the natural logical key. Every Brazilian fund has a unique CNPJ. ANBIMA fields carry the industry taxonomy for the fund's strategy.

**balances:** funds are measured in *quotas*, not shares. `quotaGrossPriceValue` is the price of one quota on `referenceDate`. `grossAmount ≈ quotaQuantity × quotaGrossPriceValue` (identity check for us).


In [19]:
print_payload('detail', sample_position_payload('FUNDS', 'detail'))

--- detail ---
{
  "name": "SPX NIMITZ ESTRUTURAL FIC FIM",
  "cnpjNumber": "34177667000185",
  "anbimaCategory": "MULTIMERCADO"
}


### 3.3 · `BANK_FIXED_INCOMES`: CDB, LCI, LCA, LC, LF

**Spec:** [bank-fixed-incomes/1.1.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/bank-fixed-incomes/1.1.0.yml)

**detail:** richer than variable income. Includes `remuneration` (the interest formula: fixed, or indexed to CDI, SELIC, IPCA), `dueDate`, `issueDate`, `clearingCode` (a B3 identifier). These are *per-customer* securities. A CDB is issued to you specifically. The logical key therefore needs several fields: at minimum `(isinCode, issuerInstitutionCnpjNumber, dueDate, issueDate)`.

**balances:** includes `updatedUnitPrice` (marked-to-market unit value) and `purchaseUnitPrice` (what you paid). `netAmount` accounts for accrued income tax (`incomeTax`). Also carries `postFixedIndexerPercentage`, which is how much of the indexer you get (for example, 102% of CDI).


In [20]:
print_payload('detail', sample_position_payload('BANK_FIXED_INCOMES', 'detail'))

--- detail ---
{
  "issuerInstitutionCnpjNumber": "60746948000112",
  "isinCode": "BRBANKLCA3A3",
  "investmentType": "LCA",
  "remuneration": {
    "rateType": "EXPONENCIAL",
    "ratePeriodicity": "DIARIO",
    "calculation": "DIAS_UTEIS",
    "indexer": "SELIC",
    "postFixedIndexerPercentage": "1.020000"
  },
  "issueUnitPrice": {
    "amount": "1000.00",
    "currency": "BRL"
  },
  "dueDate": "2024-04-01",
  "issueDate": "2021-04-05",
  "clearingCode": "LCA60746948",
  "purchaseDate": "2021-04-05",
  "gracePeriodDate": "2024-04-01"
}


### 3.4 · `CREDIT_FIXED_INCOMES`: CRI, CRA, Debêntures

**Spec:** [credit-fixed-incomes/1.1.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/credit-fixed-incomes/1.1.0.yml)

The shape mirrors bank fixed income, plus `debtorCnpjNumber` and `debtorName` for the *ultimate* borrower. For a debênture, that is the company. For a CRI, the securitizer. Logical key candidate: `(isinCode, debtorCnpjNumber, dueDate)`.

**Tier-1 defect to watch:** the sample below likely shows `issuerInstitutionCnpjNumber` ending in `.00`. That is the *tax-id decimal tail* from the case study, showing up in real synthetic data. The raw contract must coerce it.


In [21]:
print_payload('detail', sample_position_payload('CREDIT_FIXED_INCOMES', 'detail'))

--- detail ---
{
  "issuerInstitutionCnpjNumber": "92894922000108.00",
  "isinCode": "BRDEBSEC01A1",
  "investmentType": "DEBENTURES",
  "debtorCnpjNumber": "09149503000106",
  "debtorName": "OMEGA ENERGIA S.A.",
  "taxExemptProduct": "NAO",
  "remuneration": {
    "rateType": "EXPONENCIAL",
    "ratePeriodicity": "ANUAL",
    "calculation": "DIAS_UTEIS",
    "indexer": "CDI",
    "postFixedIndexerPercentage": "1.020000"
  },
  "issueUnitPrice": {
    "amount": "1000.00",
    "currency": "BRL"
  },
  "issueDate": "2022-08-06",
  "dueDate": "2029-06-15",
  "voucherPaymentIndicator": "SIM",
  "voucherPaymentPeriodicity": "SEMESTRAL",
  "clearingCode": "DEB01091495",
  "purchaseDate": "2023-03-04"
}


### 3.5 · `TREASURE_TITLES`: Tesouro Direto (government bonds)

**Spec:** [treasure-titles/1.1.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/treasure-titles/1.1.0.yml)

**detail:** a standardized government product. `isinCode` alone is a strong logical key. Every Tesouro title has a unique national ISIN. `productName` is human-readable ("Tesouro Prefixado 2027", "Tesouro IPCA+ 2035", ...).

**balances:** `updatedUnitPrice` reflects the marked-to-market bond value (it moves with rates). `purchaseUnitPrice` is what the customer paid. Difference × `quantity` approximates unrealized gain.


In [22]:
print_payload('detail', sample_position_payload('TREASURE_TITLES', 'detail'))

--- detail ---
{
  "isinCode": "BRSTNCLTN8C3",
  "productName": "Tesouro Prefixado 2027",
  "remuneration": {
    "indexer": "PRE_FIXADO",
    "ratePeriodicity": "ANUAL",
    "calculation": "DIAS_UTEIS",
    "preFixedRate": "0.112500"
  },
  "dueDate": "2027-01-01",
  "purchaseDate": "2026-03-12",
  "voucherPaymentIndicator": "NAO"
}


## 4 · Open the JSON: transaction payloads

Transactions arrive pre-flattened: one row per movement, not one row per API page. All five families share a broadly similar shape: `type`, `transactionType`, `transactionDate`, `transactionQuantity`, `transactionUnitPrice`, `transactionValue` or `transactionNetValue`, `transactionId`. Each family adds its own extras.

Note the enums:
- **`type`:** direction. `ENTRADA` means money or quantity in. `SAIDA` means out. Universal across families.
- **`transactionType`:** the semantic action. `COMPRA`, `APLICACAO`, `RESGATE`, `RENDIMENTO`, `JUROS`, `AMORTIZACAO`, and more. Per-family enum in `EnumXxxTransactionsTransactionType`.

**Consumption layer implication:** the wealth page needs a *unified* type. User-facing categories are `buy`, `sell`, `income`, `transfer`. That mapping table lives in the consumption layer (or a shared contract), not in canonical.


In [23]:
for investment_type in ['VARIABLE_INCOMES', 'BANK_FIXED_INCOMES']:
    print_payload(investment_type, sample_transaction_payload(investment_type))
    print()

--- VARIABLE_INCOMES ---
{
  "type": "ENTRADA",
  "transactionType": "COMPRA",
  "transactionDate": "2020-04-06",
  "priceFactor": "1.00000000",
  "transactionQuantity": "6301.00000000",
  "transactionUnitPrice": {
    "amount": "25.00000000",
    "currency": "BRL"
  },
  "transactionValue": {
    "amount": "157525.0000",
    "currency": "BRL"
  },
  "brokerNoteId": "75644578",
  "transactionId": "826e8d3e-1745-5e01-a161-c478a02917b8"
}

--- BANK_FIXED_INCOMES ---
{
  "type": "ENTRADA",
  "transactionType": "APLICACAO",
  "transactionDate": "2021-04-05",
  "transactionUnitPrice": {
    "amount": "1000.00000000",
    "currency": "BRL"
  },
  "transactionQuantity": "18.96700000",
  "transactionGrossValue": {
    "amount": "18967.0000",
    "currency": "BRL"
  },
  "transactionNetValue": {
    "amount": "18967.0000",
    "currency": "BRL"
  },
  "remunerationTransactionRate": "0.112500",
  "indexerPercentage": "1.020000",
  "transactionId": "0fb536fa-abb3-55c3-a5c6-49be166ddcf6"
}



In [24]:
# What transactionType values actually appear per family?
sql("""
  SELECT investment_type,
         json_extract_string(transaction_json, '$.transactionType') AS transaction_type,
         count(*)                                                   AS record_count
  FROM raw_transactions
  GROUP BY investment_type, transaction_type
  ORDER BY investment_type, record_count DESC
""")

,investment_type,transaction_type,record_count
0,BANK_FIXED_INCOMES,APLICACAO,87517
1,BANK_FIXED_INCOMES,RESGATE,75233
2,BANK_FIXED_INCOMES,OUTROS,20215
3,BANK_FIXED_INCOMES,PAGAMENTO_JUROS,8928
4,BANK_FIXED_INCOMES,VENCIMENTO,2407
5,BANK_FIXED_INCOMES,TRANSFERENCIA_CUSTODIA,1473
6,BANK_FIXED_INCOMES,AMORTIZACAO,1376
7,BANK_FIXED_INCOMES,TRANSFERENCIA_TITULARIDADE,447
8,CREDIT_FIXED_INCOMES,COMPRA,9785
9,CREDIT_FIXED_INCOMES,VENDA,2343


## 5 · A first pass at the defects

We now know the actual field names. Wire up the three named defect probes. Keep results here. They feed directly into `design/data_quality.md`.


### 5.1 · Tier-1: tax-id decimal tail (case study, named)

Tax IDs (CNPJ, CPF) are 14 or 11 digit strings. Any value with `.` in it is the exact defect the case study warns about.


In [25]:
sql("""
  SELECT investment_type,
         count(*) FILTER (WHERE json_extract_string(payload_json, '$.data.issuerInstitutionCnpjNumber') LIKE '%.%') AS issuer_cnpj_decimal_tail_count,
         count(*) FILTER (WHERE json_extract_string(payload_json, '$.data.debtorCnpjNumber')           LIKE '%.%') AS debtor_cnpj_decimal_tail_count,
         count(*)                                                                                                  AS record_count
  FROM raw_positions
  WHERE payload_kind = 'detail'
  GROUP BY investment_type
  ORDER BY investment_type
""")

,investment_type,issuer_cnpj_decimal_tail_count,debtor_cnpj_decimal_tail_count,record_count
0,BANK_FIXED_INCOMES,1370,0,64535
1,CREDIT_FIXED_INCOMES,306,0,5447
2,FUNDS,0,0,8355
3,TREASURE_TITLES,0,0,9138
4,VARIABLE_INCOMES,0,0,16479


### 5.2 · Tier-2: intra-sync duplicate

For variable income (easiest logical key), check: same `(snapshot_id, account_id, isinCode, ticker)` under more than one `investment_id`.


In [26]:
sql("""
  WITH variable_income_details AS (
    SELECT snapshot_id, account_id, investment_id,
           json_extract_string(payload_json, '$.data.isinCode') AS isin,
           json_extract_string(payload_json, '$.data.ticker')   AS ticker
    FROM raw_positions
    WHERE investment_type = 'VARIABLE_INCOMES' AND payload_kind = 'detail'
  )
  SELECT snapshot_id, account_id, isin, ticker,
         count(DISTINCT investment_id)     AS distinct_investment_id_count,
         array_agg(DISTINCT investment_id) AS investment_ids
  FROM variable_income_details
  GROUP BY snapshot_id, account_id, isin, ticker
  HAVING count(DISTINCT investment_id) > 1
  ORDER BY distinct_investment_id_count DESC
  LIMIT 5
""")

,snapshot_id,account_id,isin,ticker,distinct_investment_id_count,investment_ids
0,587debfe-73f7-5d8c-ac73-0563778ec4d6,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6420e29e-a891-5cb9-859e-696da03e7a31, 7..."
1,77895887-b582-570c-9ded-0923346c4ff3,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6420e29e-a891-5cb9-859e-696da03e7a31, 7..."
2,4c01193b-999d-5435-8fee-d3470cd7670c,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[3758c1da-6799-507a-acd8-2d6c9f65cfdc, f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6..."
3,39dff561-4127-5b01-9c14-eb1d28c1dbc4,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6420e29e-a891-5cb9-859e-696da03e7a31, 7..."
4,c24c41cc-fded-5da5-9a71-d0aa294c91f6,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[3758c1da-6799-507a-acd8-2d6c9f65cfdc, f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6..."


### 5.3 · Tier-2: identity churn

Same logical key across snapshots under more than one `investment_id`. Any hits mean the institution is violating the OFB spec's `investmentId` reuse mandate.


In [27]:
sql("""
  WITH variable_income_details AS (
    SELECT institution_id, account_id, investment_id,
           json_extract_string(payload_json, '$.data.isinCode') AS isin,
           json_extract_string(payload_json, '$.data.ticker')   AS ticker
    FROM raw_positions
    WHERE investment_type = 'VARIABLE_INCOMES' AND payload_kind = 'detail'
  )
  SELECT institution_id, account_id, isin, ticker,
         count(DISTINCT investment_id)     AS distinct_investment_id_count,
         array_agg(DISTINCT investment_id) AS investment_ids
  FROM variable_income_details
  GROUP BY institution_id, account_id, isin, ticker
  HAVING count(DISTINCT investment_id) > 1
  ORDER BY distinct_investment_id_count DESC
  LIMIT 5
""")

,institution_id,account_id,isin,ticker,distinct_investment_id_count,investment_ids
0,00000000000006,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRITUBACNPR1,ITUB4,5,"[3758c1da-6799-507a-acd8-2d6c9f65cfdc, f50c6b48-f5e8-59a6-9959-d7098dd9ef85, 7fe1bc12-1388-5abe-9840-f222b6743abf, 6..."
1,00000000000002,2ab892a8-1694-5b17-a13f-eb758d48210c,BRWEGEACNOR0,WEGE3,4,"[6073627565773498, 4771988618428728, 0742083929654326, 4159843754026982]"
2,00000000000004,ccdc8073-a0e3-5799-992d-9af6310ba197,BRBPACUNT002,BPAC11,4,"[5352657126479291, 4089541970803799, 1185747893514315, 9933514857547071]"
3,00000000000001,6b8fcd56-8159-51fd-820d-8aa84292931c,BRHASHCTF001,HASH11,4,"[ec34df5a-d547-5bab-b6f4-4eb7722d6fc2, a48b86d8-75e5-5980-bf82-4ae35e1e869f, 4a78e558-78e4-5a9b-a879-a1ddc2eed7ca, 3..."
4,00000000000006,36c13618-4ea4-5208-bf90-8b9ce29d3b32,BRVALEACNOR0,VALE3,4,"[88bebc82-9713-53fa-be23-9c615abe290b, b356be12-248c-5f24-8d9d-c69518e47a08, 8bb8e97c-6970-558e-95af-5dcfd3e78458, 8..."


### 5.4 · Tier-2: zero-transient

Balance equals 0 sandwiched between non-zero values, with quantity unchanged.


In [28]:
sql("""
  WITH balance_snapshots AS (
    SELECT party_id, investment_id, snapshot_created_at,
           cast(json_extract_string(payload_json, '$.data.quantity')            AS DOUBLE) AS quantity,
           cast(json_extract_string(payload_json, '$.data.grossAmount.amount')  AS DOUBLE) AS gross_amount
    FROM raw_positions
    WHERE payload_kind = 'balances'
  ),
  balance_snapshots_with_neighbors AS (
    SELECT *,
           lag(gross_amount) OVER holding_over_time  AS previous_gross_amount,
           lead(gross_amount) OVER holding_over_time AS next_gross_amount,
           lag(quantity) OVER holding_over_time      AS previous_quantity,
           lead(quantity) OVER holding_over_time     AS next_quantity
    FROM balance_snapshots
    WINDOW holding_over_time AS (PARTITION BY party_id, investment_id ORDER BY snapshot_created_at)
  )
  SELECT party_id, investment_id, snapshot_created_at, quantity, gross_amount,
         previous_gross_amount, next_gross_amount
  FROM balance_snapshots_with_neighbors
  WHERE gross_amount          = 0
    AND previous_gross_amount > 0
    AND next_gross_amount     > 0
    AND quantity              = previous_quantity
    AND quantity              = next_quantity
  ORDER BY snapshot_created_at
  LIMIT 5
""")

,party_id,investment_id,snapshot_created_at,quantity,gross_amount,previous_gross_amount,next_gross_amount
0,679aa92e-f12b-5860-9d15-8a2838bf3e3f,2841986982285407,2026-08-06T10:04:11Z,1589.0,0.0,75199.74,76177.93
1,679aa92e-f12b-5860-9d15-8a2838bf3e3f,5667609725231687,2026-08-06T10:04:11Z,5376.0,0.0,212734.77,216209.82
2,5a5c2d94-f9d9-5287-9baa-1b25e0030872,9596c2ee-3b47-5dc9-a6fd-ff0c9688070b,2026-08-07T09:22:31Z,5778.0,0.0,227616.22,229881.20
3,43037b26-0d10-577f-b627-b17d7bdd4c30,2351446269028886,2026-08-07T09:58:31Z,118.0,0.0,1183.35,1222.01
4,43037b26-0d10-577f-b627-b17d7bdd4c30,4880537593578090,2026-08-07T09:58:31Z,2059.0,0.0,110051.49,115007.09


## 6 · Scratch-pad: observations for the design docs

Fill this in as you go. These bullets become `design/decisions.md` and `design/data_quality.md`.

- **Grain of positions:** one snapshot × one holding gives **two rows** (balances and detail). Canonical `positions` should be the join of the two, keyed on `(party_id, account_id, snapshot_id, investment_id)` at natural grain.
- **Watermark:** `ingested_at` on both files. `snapshot_created_at` is the event time. The gap between them is the pipeline lag.
- **Logical key per family.** Record what actually works when you probed above.
  - `VARIABLE_INCOMES`: `(isinCode, ticker)`. Check that the intra-sync dup query returned 0.
  - `FUNDS`: `(cnpjNumber)`. Write and run.
  - `BANK_FIXED_INCOMES`: `(isinCode, issuerInstitutionCnpjNumber, dueDate, issueDate)`. Write and run.
  - `CREDIT_FIXED_INCOMES`: `(isinCode, debtorCnpjNumber, dueDate)`. Write and run.
  - `TREASURE_TITLES`: `(isinCode)`. Write and run.
- **Tier-1 defects observed:** fill in. Decimal-tail count, missing-required count, enum violations.
- **Tier-2 defects observed:** fill in. Intra-sync dup count per family, identity churn count per family, zero-flap count.
- **Transaction type mapping:** list the distinct values seen in the query above. Group them into wealth-consumer categories: `buy`, `sell`, `income`, `transfer`, `tax`, `other`.
